# Recursive lossless graph dictionary v2 — ConceptNet 100k

Тонкий Colab-runner для `grammar_exact_v2`: Wishart discovery → persistent graph dictionary → exact transition archives → recursive coarsening → grammar-induced finite graphon/graphex projection.

**Runtime:** рекомендуется GPU L4. Все тяжёлые вычисления и случайный I/O выполняются в `/content`; Google Drive используется для входа, checkpoint/resume и финальных артефактов.


In [ ]:
# ===== USER PARAMETERS =====
REPO_URL = 'https://github.com/SemanticMap/semgraphex.git'
BRANCH = 'feature/recursive-lossless-graph-dictionary-v2'
REPO_ROOT = '/content/semgraphex'

DRIVE_ROOT = '/content/drive/MyDrive/SemanticMap/colab/wishart'
RUN_NAME = 'wishart-grammar-v2-cn100k-01'
RUN_MODE = 'NEW'  # NEW | RESUME

# Set exactly one. Relative paths are resolved under DRIVE_ROOT.
PREPARED_DRIVE = ''
DATASET_DRIVE = ''

DEVICE = 'auto'       # auto | cuda | cpu
CPU_WORKERS = 4
BLAS_THREADS = 1

assert RUN_MODE in {'NEW', 'RESUME'}
assert bool(PREPARED_DRIVE) ^ bool(DATASET_DRIVE), 'Set exactly one of PREPARED_DRIVE or DATASET_DRIVE'


In [ ]:
# ===== RUNTIME PREFLIGHT =====
import os, platform, shutil, subprocess, sys

print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('CPU count:', os.cpu_count())
disk = shutil.disk_usage('/content')
print('Local /content free GiB:', round(disk.free / 2**30, 2))
try:
    import psutil
    print('RAM total GiB:', round(psutil.virtual_memory().total / 2**30, 2))
except Exception:
    pass
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], check=False)
else:
    print('GPU: not detected')


In [ ]:
# ===== MOUNT DRIVE =====
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ===== CHECKOUT + REPRODUCIBLE INSTALL =====
from pathlib import Path

repo = Path(REPO_ROOT)
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(repo), 'reset', '--hard', f'origin/{BRANCH}'], check=True)

sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Git SHA:', sha)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(repo / 'requirements/constraints-colab.txt'), '-e', f'{repo}[dev,wishart,gpu]'], check=True)


In [ ]:
# ===== BUILD CLI COMMAND =====
config = Path(REPO_ROOT) / 'configs/wishart_conceptnet_grammar_v2.yaml'
cmd = [
    sys.executable, '-m', 'semmap_haken.wishart_colab_cli',
    '--config', str(config),
    '--drive-root', DRIVE_ROOT,
    '--scratch-root', '/content/semmap-wishart-grammar-v2',
    '--run-name', RUN_NAME,
    '--device', DEVICE,
    '--cpu-workers', str(CPU_WORKERS),
    '--blas-threads', str(BLAS_THREADS),
]
if PREPARED_DRIVE:
    cmd += ['--prepared-drive', PREPARED_DRIVE]
else:
    cmd += ['--dataset-drive', DATASET_DRIVE]
if RUN_MODE == 'RESUME':
    cmd += ['--resume']

print('Command:', ' '.join(cmd))


In [ ]:
# ===== RUN / RESUME =====
subprocess.run(cmd, cwd=REPO_ROOT, check=True)


In [ ]:
# ===== VALIDATE PUBLISHED ARTIFACTS =====
import json

run = Path(DRIVE_ROOT) / 'runs' / RUN_NAME
assert (run / 'COMPLETED').is_file(), f'Missing COMPLETED: {run}'
colab = json.loads((run / 'COLAB_RUN.json').read_text())
hierarchy = json.loads((run / 'hierarchy.json').read_text())
print('Drive run:', run)
print('Stop reason:', hierarchy['stop_reason'])
print('Levels:', hierarchy['levels'])
print('Nodes:', hierarchy['initial_nodes'], '->', hierarchy['final_nodes'])

transitions = sorted(run.glob('transition_???_???'))
rows = []
for transition in transitions:
    metrics_path = transition / 'dictionary_metrics.json'
    if not metrics_path.is_file():
        continue
    metrics = json.loads(metrics_path.read_text())
    exact = metrics.get('exact_transition_codec') or {}
    rows.append({
        'transition': transition.name,
        'selected': metrics.get('selected_occurrences'),
        'scan_mode': metrics.get('frequency_scan_mode'),
        'reused_centers': metrics.get('incremental_reused_centers'),
        'objective': metrics.get('selection_objective'),
        'archive_bytes': exact.get('archive_bytes'),
        'baseline_json_bytes': exact.get('baseline_json_bytes'),
        'baseline_binary_bytes': exact.get('baseline_binary_bytes'),
        'ratio_json': exact.get('compression_ratio'),
        'ratio_binary': exact.get('compression_ratio_binary_baseline'),
        'roundtrip': exact.get('roundtrip_exact'),
        'adjacency_from_relations': (exact.get('adjacency_from_relations') or {}).get('matches'),
    })
for row in rows:
    print(row)
assert all(row['roundtrip'] is True for row in rows), 'A transition archive failed exact roundtrip'

hierarchy_codec = json.loads((run / 'hierarchy_codec_report.json').read_text())
assert hierarchy_codec['roundtrip_exact'] is True
print('Hierarchy exact codec:', hierarchy_codec)
assert hierarchy_codec['compression_ratio_binary_bundle'] is not None

compression = json.loads((run / 'compression_analysis.json').read_text())
print('Compression summary:', compression['summary'])
print('Hierarchy / compact binary bundle:', compression['summary']['hierarchy_compression_ratio_binary_bundle'])
print('First transition not beating binary baseline:', compression['summary']['first_transition_not_beating_binary_baseline'])

multiscale = json.loads((run / 'multiscale_graph_model.json').read_text())
print('Multiscale projections:', multiscale['projection_count'])
for distance in multiscale['adjacent_distances']:
    print('distance', distance)

print('Validated exact transition archives:', len(rows))
print('Consolidated final-to-level0 roundtrip: OK')


## Интерпретация

`compression_ratio_binary_baseline < 1` означает реальный выигрыш exact transition archive относительно компактного exact non-grammar baseline с varint-полями и bit-exact float64. Для итоговой рекурсивной модели используйте `hierarchy_compression_ratio_binary_bundle`: baseline содержит те же level-0 memberships и тот же adjacency fallback, поэтому сравнение является apples-to-apples. Старый `compression_ratio` к JSON baseline сохранён только для преемственности экспериментов. `mdl_gain_bits_proxy` и `grammar_v2_logical` — критерии выбора кандидатов, а не замена измеренному размеру файла. `dictionary_graphex.json` — конечная эмпирическая block-аппроксимация по grammar symbols с массой исходных узлов; она не является доказательством сходимости к предельному графексу.
